# 조3조 기말 프로젝트 — 지역·기온별 전력 패턴 분석

**과목:** 파이썬데이터분석 | **교수:** 서지훈  
**조원:** 202004124 하유빈, 202204173 곽소민, 202384068 황준연, 202304226 박선준, 202301099 공지수

이 노트북은 `data/` 샘플만으로 실행됩니다. 전체 코드·데이터는 GitHub 레포 `pipelines/seoul/` 참고.

## 1. 데이터 로드

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('.').resolve()
DATA = ROOT / 'data'
RESULTS = ROOT / 'results'

raw = pd.read_csv(DATA / 'sample_01_raw_kpx.csv', parse_dates=['date'])
merged = pd.read_csv(DATA / 'sample_02_merged_hourly.csv', parse_dates=['datetime'])
district = pd.read_csv(DATA / 'sample_03_seoul_district_monthly.csv')

print('원본 KPX:', raw.shape, raw.columns.tolist())
print('병합 hourly:', merged.shape)
print('서울 구별:', district.shape)
display(raw.head(3))
display(merged.head(3))

## 2. Track 1 — 4지역 시간대별 패턴 (샘플 1주)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True, sharey=False)
regions = ['서울시', '부산시', '대전시', '강원도']
for ax, reg in zip(axes.flat, regions):
    sub = merged[merged['region'] == reg]
    hourly = sub.groupby('hour')['power_mwh'].mean()
    ax.plot(hourly.index, hourly.values, marker='o', markersize=3)
    ax.set_title(reg)
    ax.set_xlabel('시간 (0~23)')
    ax.set_ylabel('평균 전력거래량 (MWh)')
    ax.grid(True, alpha=0.3)
plt.suptitle('Track 1: 지역별 시간대 패턴 (2024-01-01~07 샘플)', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Track 1 — 기온 vs 전력 (U자 확인, 샘플)

In [ ]:
seoul = merged[merged['region'] == '서울시']
plt.figure(figsize=(8, 5))
plt.scatter(seoul['temp_c'], seoul['power_mwh'], alpha=0.4, s=15)
plt.xlabel('기온 (°C)')
plt.ylabel('전력거래량 (MWh)')
plt.title('서울: 기온 × 전력거래량 (샘플 1주)')
plt.grid(True, alpha=0.3)
plt.show()

## 4. Track 2 — 서울 구별 월별 사용량 (상위 10구)

In [ ]:
annual = district.groupby('district')['usage'].sum().sort_values(ascending=False)
top10 = annual.head(10)
plt.figure(figsize=(10, 5))
top10.plot(kind='barh', color='steelblue')
plt.xlabel('연간 합계 사용량')
plt.title('서울 구별 에너지 사용량 Top 10 (월별 데이터 합산)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()
print(f'1위 {top10.index[0]} / 10위 {top10.index[9]} = {top10.iloc[0]/top10.iloc[9]:.1f}배 차이')

## 5. Track 3 — 모델 검증 결과 (2024, 전체 데이터 기준)

샘플 데이터로 모델을 다시 학습하지 않고, **미리 계산해 둔 검증 결과**를 표시합니다.

In [ ]:
with open(RESULTS / 'backtest_summary_2020_2024.json', encoding='utf-8') as f:
    backtest = json.load(f)

rows = []
for year, v in sorted(backtest['years'].items()):
    rows.append({
        '연도': year,
        '방법': v['method'],
        '평균오차_MWh': v.get('hybrid'),
        '비고': v.get('note', '')[:40],
    })
df_bt = pd.DataFrame(rows)
display(df_bt)

y2024 = backtest['years']['2024']
print('\n=== 2024 핵심 (전체 연도 out-of-sample 검증) ===')
print(f"  4일 한 번에 예측:  평균 오차 {y2024['ttm']} MWh")
print(f"  하루씩 이어 맞추기: 평균 오차 {y2024['gru']} MWh")
print(f"  → 서울 시간당 ~600 MWh 기준, 약 {y2024['ttm']/600*100:.0f}% 수준")

## 6. 결론

1. **Track 1:** 지역마다 시간대·기온 반응 패턴이 다름 (발전 vs 소비 구조)
2. **Track 2:** 서울 내부에서도 구별 에너지 사용량 차이 큼
3. **Track 3:** 시계열 AI + 서울 맞춤 예측기로 2024 검증 — 4일 한 번에 예측이 하루씩 이어보다 안정적

**전체 코드·재현:** GitHub + `pipelines/seoul/README.md`  
**모델 가중치:** `models/hybrid_seoul.pt` (TTM 본체는 HuggingFace 별도)